# Class NN: Would It Replicate?
*Elements of Data Science — Honors*

**First thing:** save this notebook under a new name that includes your team's name.

Part 1 is on paper. This notebook is Parts 2–4. Keep the worksheet next to you and
**write each prediction on the worksheet before you run the cell it is about.**

In [ ]:
from datascience import *
import numpy as np
# import for plotting
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
# Fix for datascience plots
import collections as collections
import collections.abc as abc
collections.Iterable = abc.Iterable

---
## The Creek

Everything in this notebook is **simulated**. The creek, the sampling sites and every chloride
value come out of a random number generator. It is built to look like a Pennypack-style
upstream/downstream chloride survey, but it is **not** the Pennypack data you analyzed, and
nothing here tells you anything about the real Pennypack.

Because we built this creek, we know its truth, which you never get with real data:

- Upstream sites average **150 mg/L** chloride; downstream sites average **183 mg/L**.
  The true downstream excess is **33 mg/L**.
- Individual sites scatter around their average with an SD of **40 mg/L**.
- Every survey samples **12 sites upstream and 12 sites downstream**.

The statistic is the same one you used before: **downstream mean minus upstream mean**.

---
## Part 2. Run the Creek

**2.1** One survey of the creek. `true_difference` is the downstream excess built into
the creek. The function returns the difference in means that one survey *observes*.

In [ ]:
def sample_creek(true_difference):
    upstream = np.random.normal(150, 40, 12)
    downstream = np.random.normal(150 + true_difference, 40, 12)
    return np.mean(downstream) - np.mean(upstream)

sample_creek(33)

Run that cell a few times. The truth is 33 every time; the survey is not.

**2.2** The null distribution. A p-value asks: *if there were no downstream excess at all, how
often would a survey show a difference at least this big, in either direction?*

In Class 15 you answered that by shuffling the labels of one dataset. Here we know exactly
how the creek works, so we can do something simpler: survey a creek whose true difference is
**0** ten thousand times. We build this **once** and use it for every p-value in the notebook.
It takes a few seconds.

In [ ]:
null_diffs = make_array()
for i in np.arange(10000):
    null_diffs = np.append(null_diffs, sample_creek(0))

print('null surveys:', len(null_diffs), '  (should be 10000)')
Table().with_column('difference (mg/L)', null_diffs).hist(bins=np.arange(-70, 71, 5))
plt.title('10,000 surveys of a creek with NO downstream excess');

The p-value of any observed difference is the fraction of these 10,000 null surveys that
were at least as far from zero.

In [ ]:
def p_value_of(observed):
    as_extreme = np.count_nonzero(np.abs(null_diffs) >= np.abs(observed))
    return as_extreme / len(null_diffs)

**2.3** One complete survey: the difference it observed and its p-value. Run this cell three
times and record all three on the worksheet (question 3).

In [ ]:
one_difference = sample_creek(33)
print('observed difference:', np.round(one_difference, 1), 'mg/L')
print('p-value:            ', p_value_of(one_difference))

**2.4** Many surveys. Our rule: every loop runs at 10 before it runs big. Run this at 10 and
look at the ten p-values. Then **write your prediction for question 4**, change the 10 to 1000,
and run it again.

In [ ]:
p_values = make_array()
for i in np.arange(10):
    p_values = np.append(p_values, p_value_of(sample_creek(33)))

Table().with_column('p-value', p_values).show(10)

**2.5** The picture. The p-values run from tiny to large, so the x-axis is on a **log scale**:
each tick is ten times the one before. The red line is 0.05.

Our null has 10,000 surveys, so the smallest p-value it can measure is 1/10,000 = 0.0001. A survey
more extreme than every null survey gets p = 0, which a log axis can't show; those are plotted at
0.0001.

In [ ]:
shown = np.maximum(p_values, 0.0001)

plt.hist(shown, bins=10 ** np.arange(-4, 0.01, 0.5))
plt.xscale('log')
plt.axvline(0.05, color='red')
plt.xlabel('p-value (log scale)')
plt.ylabel('number of surveys')
plt.title(str(len(p_values)) + ' surveys of the same creek');

In [ ]:
print('surveys:              ', len(p_values))
print('fraction with p < 0.05:', np.count_nonzero(p_values < 0.05) / len(p_values))
print('smallest p-value:      ', min(p_values))
print('largest p-value:       ', max(p_values))

---
## Part 3. Only the Surveys Like Team A's

**3.1** Now the creek runs in **pairs**. Each pair is an *original* survey and a *replicate*:
the exact same survey done again with new random sites. Run at 10, then change the 10 to 1000.

In [ ]:
originals = make_array()
replicates = make_array()
for i in np.arange(10):
    originals = np.append(originals, p_value_of(sample_creek(33)))
    replicates = np.append(replicates, p_value_of(sample_creek(33)))

pairs = Table().with_columns('original p', originals, 'replicate p', replicates)
pairs.show(5)
print('pairs:', pairs.num_rows)

**3.2** Team A got p = 0.03. Keep only the pairs whose *original* came out like that: at
least 0.01 and less than 0.05. `are.between(a, b)` keeps values with a ≤ value < b.
Fill in the two numbers.

**Before you run it**, answer question 5 on the worksheet.

In [ ]:
like_team_a = pairs.where('original p', are.between(..., ...))

print('pairs kept:', like_team_a.num_rows, 'of', pairs.num_rows)
print()
print('fraction of THEIR replicates with p < 0.05:',
      np.count_nonzero(like_team_a.column('replicate p') < 0.05) / like_team_a.num_rows)
print('fraction of ALL replicates with p < 0.05:  ',
      np.count_nonzero(pairs.column('replicate p') < 0.05) / pairs.num_rows)

**3.3** Why that number? A survey gets p < 0.05 when its observed difference lands farther
from zero than 95% of the null surveys. Here is that distance:

In [ ]:
cutoff = percentile(95, np.abs(null_diffs))
cutoff

Use this and the creek's true difference (33 mg/L) to answer question 7.

---
## Part 4. Twenty Surveys, Twenty Intervals

This cell is prebuilt. **You do not need to read the code.** It runs twenty surveys of the same
creek. For each one it computes the observed difference, a 95% bootstrap confidence interval (the
way you made them in Class 19: resample each side with replacement, 1000 times, and take the middle
95% of the differences), and the p-value from your null distribution.

Every team gets the same twenty surveys, so the class can talk about the same picture. Your
p-values may differ from another team's in the third decimal place, because each team built its
own null distribution.

In [ ]:
# ---- prebuilt: you do not need to read this ----
np.random.seed(120)
differences = make_array()
lowers = make_array()
uppers = make_array()
p_values_20 = make_array()

for s in np.arange(20):
    upstream = np.random.normal(150, 40, 12)
    downstream = np.random.normal(183, 40, 12)
    observed = np.mean(downstream) - np.mean(upstream)
    boot = make_array()
    for i in np.arange(1000):
        up_resample = np.random.choice(upstream, 12)
        down_resample = np.random.choice(downstream, 12)
        boot = np.append(boot, np.mean(down_resample) - np.mean(up_resample))
    differences = np.append(differences, observed)
    lowers = np.append(lowers, percentile(2.5, boot))
    uppers = np.append(uppers, percentile(97.5, boot))
    p_values_20 = np.append(p_values_20, p_value_of(observed))
np.random.seed(None)

plt.figure(figsize=(9, 8))
for s in np.arange(20):
    if p_values_20.item(s) < 0.05:
        color = 'tab:blue'
    else:
        color = 'darkgray'
    plt.plot([lowers.item(s), uppers.item(s)], [s + 1, s + 1], color=color, linewidth=4)
    plt.plot(differences.item(s), s + 1, 'o', color='black', markersize=5)
    if p_values_20.item(s) < 0.001:
        label = 'p < 0.001'
    else:
        label = 'p = ' + str(np.round(p_values_20.item(s), 3))
    plt.text(108, s + 1, label, va='center', fontsize=11)
plt.axvline(0, color='black', linestyle=':', linewidth=1.5)
plt.axvline(33, color='red', linewidth=1.5)
plt.xlim(-40, 140)
plt.ylim(20.7, 0.3)
plt.yticks(np.arange(1, 21))
plt.xlabel('downstream minus upstream (mg/L)')
plt.ylabel('survey')
plt.title('Same creek, 20 surveys.  Red: true difference 33.  Blue: p < 0.05', fontsize=13);

In [ ]:
surveys = Table().with_columns(
    'survey', np.arange(1, 21),
    'difference', np.round(differences, 1),
    'CI low', np.round(lowers, 1),
    'CI high', np.round(uppers, 1),
    'p-value', np.round(p_values_20, 4))
surveys.show()

Answer questions 8–10 on the worksheet from the plot and this table.

---
## If You Finish Early

**A.** Run 1000 surveys and keep the observed difference only when the survey came out
significant. Compare their average with the truth.

In [ ]:
significant_diffs = make_array()
for i in np.arange(1000):
    d = sample_creek(33)
    if p_value_of(d) < 0.05:
        significant_diffs = np.append(significant_diffs, d)

print('significant surveys:                ', len(significant_diffs))
print('their average observed difference:  ', np.round(np.mean(significant_diffs), 1), 'mg/L')
print('true difference:                     33 mg/L')

**B.** Suppose the true downstream excess were **60 mg/L** instead of 33. Copy the cells from
3.1 and 3.2 below, change `sample_creek(33)` to `sample_creek(60)` in **both** lines, and run them
at 1000. (The null distribution does not change. Why not?)